# Publications

<!-- structured-notebook -->
## Notebook Summary
Purpose: collect publication data for the project from PubMed, arXiv, and bioRxiv or medRxiv, including query construction, API paging, and CSV export steps.

Main steps:
- Describe the target publication sources and the search strategy used to retrieve records.
- Fetch records from each API with source-specific code for paging, parsing, and metadata extraction.
- Export the resulting datasets so later notebooks can analyze them.


In [ ]:
# structured-notebook-bootstrap
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists():
            return candidate
    return start


_repo_root = find_repo_root(Path.cwd())
if str(_repo_root) not in sys.path:
    sys.path.append(str(_repo_root))

from src.project_paths import (
    ARXIV_RAW_DIR,
    CHROMA_DIR,
    EXTERNAL_NEWS_DIR,
    GUARDIAN_DATA_DIR,
    LLM_CLASSIFICATION_DIR,
    NEWS_HTML_DIR,
    NEWS_OUTPUT_DIR,
    PREPRINT_RAW_DIR,
    PROQUEST_PROCESSED_DIR,
    PROQUEST_UNPROCESSED_DIR,
    PUBMED_PROCESSED_DIR,
    PUBMED_RAW_DIR,
    PUBLICATIONS_TABLE_DIR,
    REDDIT_DATA_DIR,
    ROOT,
    RQ1_FIGURES_DIR,
    RQ4_PLOTS_DIR,
    TOPIC_MATCHING_DIR,
    YOUTUBE_DATA_DIR,
)


## Smart Data Collection

https://www.sciencedirect.com/science/article/pii/S2667096821000367

<!-- structured-notebook -->
## PubMed Retrieval Note
This section defines the PubMed query and the helper code used to fetch records through Entrez. It is the main ingestion path for the biomedical publication side of the project.


## PubMed

In [5]:
from Bio import Entrez
import pandas as pd
from tqdm import tqdm
from dotenv import load_dotenv 
import os

load_dotenv() #This one is needed to fetch an Entrez registered email from local environment
Entrez.email = os.getenv("PUBMED_MAIL") 

In [4]:
import time

HEALTH_TOPICS = {
    # "Telomere": "(telomere OR telomeres OR telomerase) AND (longevity OR 'healthy aging')",
    "rapamycin": "(rapamycin) AND (longevity OR aging)",
    "metformin": "(metformin) AND (longevity OR aging)",
    "nicotinamide": "(nicotinamide) AND (longevity OR aging)",
    "resveratrol": "(resveratrol) AND (longevity OR aging)",
    "crispr" : "(crispr) AND (longevity OR aging)",
    
}


# Date ranges for comparison
DATE_RANGES = {
     "15-year data": ("2010/01/01", "2026/02/12"), # let us not divide by pre-pandemic, pandemic, and post-pandemic at this point of research
#     "pandemic": ("2020/01/01", "2022/12/31"),
#     "post_pandemic": ("2023/01/01", "2025/06/21")
}

def fetch_pubmed_data(topic_name, query, date_range, max_results=500):
    """Fetches PubMed articles with metadata and abstracts"""
    search_term  = f"({query}) AND {date_range[0]}:{date_range[1]}[PDAT]"
    
    # Step 1: Search PubMed
    handle = Entrez.esearch(db="pubmed",
                            term=search_term,
                            retmax=max_results,
                            sort="relevance",  # Get most representative articles first
                            usehistory="y") #saves the search on PubMed’s history server, so you can fetch results in pages.
    search_results = Entrez.read(handle) #reads and parses the search results
    handle.close()

    #metadata about the search
    #needed later to retreive search results without rerunning the code
    webenv = search_results["WebEnv"] #an identifier for my search session on pubMed's server
    query_key = search_results["QueryKey"] #a reference to specific query within session
    total = int(search_results["Count"]) #total number of results matching my query
    
    # print(f"Found {total} articles for {topic_name} ({date_range[0]} to {date_range[1]})")

    print(f"Found {total} articles for {topic_name}")

    # Step 2: Batch fetching by 100 articles
    batch_size = 100
    articles = []

    #loop runs from 0 up to the minimum between what the search found and the max number of results we wanted
    #tqdm is needed for graphical bar progress
    #the loop runs by fetching by 100 articles per one loop run
    for start in tqdm(range(0, min(total, max_results), batch_size)): 
        end = min(total, start + batch_size)
        handle = Entrez.efetch(db="pubmed",
                               retstart=start,
                               retmax=batch_size,
                               webenv=webenv,
                               query_key=query_key,
                               retmode="xml")
        data = Entrez.read(handle)
        articles += parse_articles(data)
        handle.close()
        time.sleep(0.3)  # Avoid overloading server
    
    return articles

def parse_articles(data):
    """Extracts key information from PubMed XML"""
    parsed = []
    for article in data['PubmedArticle']:
        try:
            pmid = article['MedlineCitation']['PMID'].strip()
            title = article['MedlineCitation']['Article']['ArticleTitle'].strip()
            
            # Abstract handling (some articles lack abstracts)
            abstract = ""
            if 'Abstract' in article['MedlineCitation']['Article']:
                abstract_parts = article['MedlineCitation']['Article']['Abstract']['AbstractText']
                abstract = " ".join([text for text in abstract_parts if isinstance(text, str)])
            
            # Journal and date info
            journal = article['MedlineCitation']['Article']['Journal']['Title']
            pub_date = article['MedlineCitation']['Article']['Journal']['JournalIssue']['PubDate']
            year = pub_date.get('Year', '')
            
            # MeSH terms for topic validation
            # Medical subject headings assigned to this article
            mesh_terms = []
            if 'MeshHeadingList' in article['MedlineCitation']:
                for item in article['MedlineCitation']['MeshHeadingList']:
                    mesh_terms.append(item['DescriptorName'])
            
            parsed.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
                "journal": journal,
                "year": year,
                "pub_date": str(pub_date),
                "mesh_terms": "; ".join(mesh_terms),
                "source": "PubMed"
            })
        except KeyError as e:
            continue
    
    return parsed

# Collect data for all combinations
all_data = []

for topic_name, query in HEALTH_TOPICS.items():
    for period, date_range in DATE_RANGES.items():
        articles = fetch_pubmed_data(f"{topic_name}_{period}", 
                                     query, 
                                     date_range,
                                     max_results=10000)
        for article in articles:
            article["topic"] = topic_name
            article["period"] = period
        all_data.extend(articles)

# Convert to DataFrame and save
df = pd.DataFrame(all_data)
df.to_csv(PUBMED_RAW_DIR / 'pubmed_centenarian_articles.csv', index=False)
print(f"Saved {len(df)} articles with {len(df) - df['abstract'].isna().sum()} abstracts")

/home/iskender/longevity_research/.venv/lib/python3.11/site-packages/Bio/Entrez/__init__.py:734: UserWarning: 
            Email address is not specified.

            To make use of NCBI's E-utilities, NCBI requires you to specify your
            email address with each request.  As an example, if your email address
            is A.N.Other@example.com, you can specify it as follows:
               from Bio import Entrez
               Entrez.email = 'A.N.Other@example.com'
            In case of excessive usage of the E-utilities, NCBI will attempt to contact
            a user at the email address provided before blocking access to the
            E-utilities.
  warnings.warn(


Found 2101 articles for nicotinamide_15-year data


100%|███████████████████████████████████████████| 15/15 [03:59<00:00, 15.99s/it]

Saved 1500 articles with 1500 abstracts


In [4]:
import time

HEALTH_TOPICS = {
    "Centenarians": "(centenarian OR centenarians OR supercentenarian OR superager OR super-ager) AND (longevity OR 'healthy aging')",
    "Senescent_cells": "('senescent cell' OR 'senescent cells') AND (longevity OR 'healthy aging')",
    "Telomere": "(telomere OR telomeres OR telomerase) AND (longevity OR 'healthy aging')",
    "rapamycin": "(rapamycin) AND (longevity OR 'healthy aging')",
    "metformin": "(metformin) AND (longevity OR aging)",
    "dementia_alzheimer": "(dementia OR alzheimer) AND (longevity OR 'healthy aging')",
    "exercise_fitness": "(exercise OR 'physical activity') AND (longevity OR 'healthy aging')",
    "diabetes_blood_sugar": "(diabetes OR 'blood sugar') AND (longevity OR 'healthy aging')",
    "gut_microbiome": "('gut microbiome') AND (longevity OR 'healthy aging')",
    "epigenetic_methylation": "(epigenetic OR methylation) AND (longevity OR 'healthy aging')",
    "mitochondria_mitophagy": "(mitochondria OR mitophagy) AND (longevity OR 'healthy aging')",
    "sirtuins": "(sirtuins OR sirt1 OR sirt3) AND (longevity OR 'healthy aging')",
    "autophagy": "(autophagy) AND (longevity OR 'healthy aging')",
    "cardiovascular_disease": "('cardiovascular disease') AND (longevity OR 'healthy aging')",
    "obesity": "(obesity) AND (longevity OR 'healthy aging')",
    "cancer": "(cancer) AND (longevity OR 'healthy aging')",
    "modafinil_nootropic": "(modafinil OR nootropic OR nootropics) AND (longevity OR 'healthy aging')",
}


# Date ranges for comparison
DATE_RANGES = {
     "15-year data": ("2010/01/01", "2025/08/24"), # let us not divide by pre-pandemic, pandemic, and post-pandemic at this point of research
#     "pandemic": ("2020/01/01", "2022/12/31"),
#     "post_pandemic": ("2023/01/01", "2025/06/21")
}

def fetch_pubmed_data(topic_name, query, date_range, max_results=500):
    """Fetches PubMed articles with metadata and abstracts"""
    search_term  = f"({query}) AND {date_range[0]}:{date_range[1]}[PDAT]"
    
    # Step 1: Search PubMed
    handle = Entrez.esearch(db="pubmed",
                            term=search_term,
                            retmax=max_results,
                            sort="relevance",  # Get most representative articles first
                            usehistory="y") #saves the search on PubMed’s history server, so you can fetch results in pages.
    search_results = Entrez.read(handle) #reads and parses the search results
    handle.close()

    #metadata about the search
    #needed later to retreive search results without rerunning the code
    webenv = search_results["WebEnv"] #an identifier for my search session on pubMed's server
    query_key = search_results["QueryKey"] #a reference to specific query within session
    total = int(search_results["Count"]) #total number of results matching my query
    
    # print(f"Found {total} articles for {topic_name} ({date_range[0]} to {date_range[1]})")

    print(f"Found {total} articles for {topic_name}")

    # Step 2: Batch fetching by 100 articles
    batch_size = 100
    articles = []

    #loop runs from 0 up to the minimum between what the search found and the max number of results we wanted
    #tqdm is needed for graphical bar progress
    #the loop runs by fetching by 100 articles per one loop run
    for start in tqdm(range(0, min(total, max_results), batch_size)): 
        end = min(total, start + batch_size)
        handle = Entrez.efetch(db="pubmed",
                               retstart=start,
                               retmax=batch_size,
                               webenv=webenv,
                               query_key=query_key,
                               retmode="xml")
        data = Entrez.read(handle)
        articles += parse_articles(data)
        handle.close()
        time.sleep(0.3)  # Avoid overloading server
    
    return articles

def parse_articles(data):
    """Extracts key information from PubMed XML"""
    parsed = []
    for article in data['PubmedArticle']:
        try:
            pmid = article['MedlineCitation']['PMID'].strip()
            title = article['MedlineCitation']['Article']['ArticleTitle'].strip()
            
            # Abstract handling (some articles lack abstracts)
            abstract = ""
            if 'Abstract' in article['MedlineCitation']['Article']:
                abstract_parts = article['MedlineCitation']['Article']['Abstract']['AbstractText']
                abstract = " ".join([text for text in abstract_parts if isinstance(text, str)])
            
            # Journal and date info
            journal = article['MedlineCitation']['Article']['Journal']['Title']
            pub_date = article['MedlineCitation']['Article']['Journal']['JournalIssue']['PubDate']
            year = pub_date.get('Year', '')
            
            # MeSH terms for topic validation
            # Medical subject headings assigned to this article
            mesh_terms = []
            if 'MeshHeadingList' in article['MedlineCitation']:
                for item in article['MedlineCitation']['MeshHeadingList']:
                    mesh_terms.append(item['DescriptorName'])
            
            parsed.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
                "journal": journal,
                "year": year,
                "pub_date": str(pub_date),
                "mesh_terms": "; ".join(mesh_terms),
                "source": "PubMed"
            })
        except KeyError as e:
            continue
    
    return parsed

# Collect data for all combinations
all_data = []

for topic_name, query in HEALTH_TOPICS.items():
    for period, date_range in DATE_RANGES.items():
        articles = fetch_pubmed_data(f"{topic_name}_{period}", 
                                     query, 
                                     date_range,
                                     max_results=1500)
        for article in articles:
            article["topic"] = topic_name
            article["period"] = period
        all_data.extend(articles)

# Convert to DataFrame and save
df = pd.DataFrame(all_data)
df.to_csv(PUBMED_RAW_DIR / 'pubmed_centenarian_articles.csv', index=False)
print(f"Saved {len(df)} articles with {len(df) - df['abstract'].isna().sum()} abstracts")

Found 1039 articles for Centenarians_15-year data


100%|███████████████████████████████████████████| 11/11 [00:32<00:00,  2.99s/it]


Found 19060 articles for Senescent_cells_15-year data


100%|███████████████████████████████████████████| 15/15 [00:49<00:00,  3.28s/it]


Found 1673 articles for Telomere_15-year data


...

Found 10357 articles for cancer_15-year data


100%|███████████████████████████████████████████| 15/15 [00:54<00:00,  3.60s/it]


Found 148 articles for modafinil_nootropic_15-year data


100%|█████████████████████████████████████████████| 2/2 [00:05<00:00,  2.51s/it]


Saved 21587 articles with 21587 abstracts


In [4]:
import time

HEALTH_TOPICS = {
    "longevity": "longevity OR aging OR healthy aging OR biohacking OR anti-aging OR health rejuventation OR living longer OR slow aging OR nootropics OR well aging", #core keywords
}

# Date ranges for comparison
DATE_RANGES = {
     "15-year data": ("2010/01/01", "2025/08/24"), # let us not divide by pre-pandemic, pandemic, and post-pandemic at this point of research
#     "pandemic": ("2020/01/01", "2022/12/31"),
#     "post_pandemic": ("2023/01/01", "2025/06/21")
}

def fetch_pubmed_data(topic_name, query, date_range, max_results=500):
    """Fetches PubMed articles with metadata and abstracts"""
    search_term  = f"({query}) AND {date_range[0]}:{date_range[1]}[PDAT]"
    
    # Step 1: Search PubMed
    handle = Entrez.esearch(db="pubmed",
                            term=search_term,
                            retmax=max_results,
                            sort="relevance",  # Get most representative articles first
                            usehistory="y") #saves the search on PubMed’s history server, so you can fetch results in pages.
    search_results = Entrez.read(handle) #reads and parses the search results
    handle.close()

    #metadata about the search
    #needed later to retreive search results without rerunning the code
    webenv = search_results["WebEnv"] #an identifier for my search session on pubMed's server
    query_key = search_results["QueryKey"] #a reference to specific query within session
    total = int(search_results["Count"]) #total number of results matching my query
    
    # print(f"Found {total} articles for {topic_name} ({date_range[0]} to {date_range[1]})")

    print(f"Found {total} articles for {topic_name}")

    # Step 2: Batch fetching by 100 articles
    batch_size = 100
    articles = []

    #loop runs from 0 up to the minimum between what the search found and the max number of results we wanted
    #tqdm is needed for graphical bar progress
    #the loop runs by fetching by 100 articles per one loop run
    for start in tqdm(range(0, min(total, max_results), batch_size)): 
        end = min(total, start + batch_size)
        handle = Entrez.efetch(db="pubmed",
                               retstart=start,
                               retmax=batch_size,
                               webenv=webenv,
                               query_key=query_key,
                               retmode="xml")
        data = Entrez.read(handle)
        articles += parse_articles(data)
        handle.close()
        time.sleep(0.3)  # Avoid overloading server
    
    return articles

def parse_articles(data):
    """Extracts key information from PubMed XML"""
    parsed = []
    for article in data['PubmedArticle']:
        try:
            pmid = article['MedlineCitation']['PMID'].strip()
            title = article['MedlineCitation']['Article']['ArticleTitle'].strip()
            
            # Abstract handling (some articles lack abstracts)
            abstract = ""
            if 'Abstract' in article['MedlineCitation']['Article']:
                abstract_parts = article['MedlineCitation']['Article']['Abstract']['AbstractText']
                abstract = " ".join([text for text in abstract_parts if isinstance(text, str)])
            
            # Journal and date info
            journal = article['MedlineCitation']['Article']['Journal']['Title']
            pub_date = article['MedlineCitation']['Article']['Journal']['JournalIssue']['PubDate']
            year = pub_date.get('Year', '')
            
            # MeSH terms for topic validation
            # Medical subject headings assigned to this article
            mesh_terms = []
            if 'MeshHeadingList' in article['MedlineCitation']:
                for item in article['MedlineCitation']['MeshHeadingList']:
                    mesh_terms.append(item['DescriptorName'])
            
            parsed.append({
                "pmid": pmid,
                "title": title,
                "abstract": abstract,
                "journal": journal,
                "year": year,
                "pub_date": str(pub_date),
                "mesh_terms": "; ".join(mesh_terms),
                "source": "PubMed"
            })
        except KeyError as e:
            continue
    
    return parsed

# Collect data for all combinations
all_data = []

for topic_name, query in HEALTH_TOPICS.items():
    for period, date_range in DATE_RANGES.items():
        articles = fetch_pubmed_data(f"{topic_name}_{period}", 
                                     query, 
                                     date_range,
                                     max_results=1500)
        for article in articles:
            article["topic"] = topic_name
            article["period"] = period
        all_data.extend(articles)

# Convert to DataFrame and save
df = pd.DataFrame(all_data)
df.to_csv(PUBMED_RAW_DIR / 'pubmed_longevity_articles_1500.csv', index=False)
print(f"Saved {len(df)} articles with {len(df) - df['abstract'].isna().sum()} abstracts")

Found 505597 articles for health_15-year data


100%|███████████████████████████████████████████| 15/15 [03:19<00:00, 13.30s/it]

Saved 1500 articles with 1500 abstracts


<!-- structured-notebook -->
## ArXiv Retrieval Note
The next block uses the arXiv API and Atom feed parsing to build a time-bounded query for healthy-aging-related records.


## ArXiv

Manual for using ArXiv API:
<br>
https://info.arxiv.org/help/api/basics.html
<br>
https://info.arxiv.org/help/api/user-manual.html
<br>
https://info.arxiv.org/help/oa/index.html

In [46]:
import urllib, urllib.request, urllib.parse, feedparser

feedparser.mixin._FeedParserMixin.namespaces[
    "http://arxiv.org/schemas/atom"
] = "arxiv"
feedparser.mixin._FeedParserMixin.namespaces[
    "http://a9.com/-/spec/opensearch/1.1/"
] = "opensearch"
base = 'http://export.arxiv.org/api/query?'

raw = 'all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging'

date = '+AND+submittedDate:[201001010000+TO+202507200000]'

max_results = 1000

query = base + 'search_query=' + raw + date +'&sortBy=relevance&sortOrder=descending&start=0&max_results=' + str(max_results)

print(query)  # check for no spaces
data = urllib.request.urlopen(query)

http://export.arxiv.org/api/query?search_query=all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging+AND+submittedDate:[201001010000+TO+202507200000]&sortBy=relevance&sortOrder=descending&start=0&max_results=1000


In [50]:
max_per_call = 200
raw = 'all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging'

records = []
for year in range(10, 26):
    url = (f"{base}search_query={raw}"
           f"+AND+submittedDate:[20{year}01010000+TO+20{year}12312359]"
           f"&start={0}&max_results={max_per_call}"
           "&sortBy=relevance&sortOrder=descending")
    print(url)
    print(f"Fetching 20{year}")
    data = urllib.request.urlopen(url).read()
    feed = feedparser.parse(data)
    if not feed.entries:
        print(f"No entries returned at year={year}; stopping.")
        break
    records.extend(feed.entries)
    time.sleep(3)  # arXiv recommends 3-second delay :contentReference[oaicite:3]{index=3}

print(f"Fetched total entries: {len(records)}")

http://export.arxiv.org/api/query?search_query=all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging+AND+submittedDate:[201001010000+TO+201012312359]&start=0&max_results=200&sortBy=relevance&sortOrder=descending
Fetching 2010
http://export.arxiv.org/api/query?search_query=all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging+AND+submittedDate:[201101010000+TO+201112312359]&start=0&max_results=200&sortBy=relevance&sortOrder=descending
Fetching 2011
http://export.arxiv.org/api/query?search_query=all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging+AND+submittedDate:[201201010000+TO+201212312359]&start=0&max_results=200&sortBy=relevance&sortOrder=descending
Fetching 2012
http://export.arxiv.org/api/query?search_query=all:longevity+OR+biohacking+OR+anti-aging+OR+healthy+aging+AND+submittedDate:[201301010000+TO+201312312359]&start=0&max_results=200&sortBy=relevance&sortOrder=descending
Fetching 2013
http://export.arxiv.org/api/query?search_query=all:longevity+OR+biohacking+O

In [52]:
records_dicts = []
for entry in records:
    rec = {
        "arxiv_id": entry.id.split("/abs/")[-1],
        "title": entry.title.strip(),
        "abstract": entry.summary.strip(),
        "published": entry.published,
        "updated": getattr(entry, "updated", ""),
        "authors": ", ".join(a.name for a in entry.authors),
        "pdf_url": next((link.href for link in entry.links if link.type=="application/pdf"), ""),
        "primary_category": entry.tags[0].term if entry.tags else "",
        "categories": ";".join([c.term for c in entry.tags]),
        "comment": getattr(entry, "arxiv_comment", ""),
        "journal_ref": getattr(entry, "arxiv_journal_ref", ""),
        "doi": getattr(entry, "arxiv_doi", ""),
        "affiliation": getattr(entry, "arxiv_affiliation", "")
    }
    records_dicts.append(rec)

df = pd.DataFrame(records_dicts)
df['published'] = pd.to_datetime(df['published'])
df['year'] = df['published'].dt.year

display(df.info())
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2520 entries, 0 to 2519
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype              
---  ------            --------------  -----              
 0   arxiv_id          2520 non-null   object             
 1   title             2520 non-null   object             
 2   abstract          2520 non-null   object             
 3   published         2520 non-null   datetime64[ns, UTC]
 4   updated           2520 non-null   object             
 5   authors           2520 non-null   object             
 6   pdf_url           2520 non-null   object             
 7   primary_category  2520 non-null   object             
 8   categories        2520 non-null   object             
 9   comment           2520 non-null   object             
 10  journal_ref       2520 non-null   object             
 11  doi               2520 non-null   object             
 12  affiliation       2520 non-null   object             
 13  yea

None

,arxiv_id,title,abstract,published,updated,authors,pdf_url,primary_category,categories,comment,journal_ref,doi,affiliation,year
0,1011.2829v1,Mathematical link of evolving aging and comple...,Aging is a fundamental aspect of living system...,2010-11-12 05:03:27+00:00,2010-11-12T05:03:27Z,"Byung Mook Weon, Jung Ho Je",http://arxiv.org/pdf/1011.2829v1,q-bio.PE,q-bio.PE;physics.bio-ph;physics.data-an,"14 pages, 5 figures, submitted to PLoS ONE",,,,2010
1,1001.5288v1,Longevity Studies in the CDF II Silicon Detector,The CDF Run II silicon detector is the largest...,2010-01-28 23:03:06+00:00,2010-01-28T23:03:06Z,Satyajit Behari,http://arxiv.org/pdf/1001.5288v1,physics.ins-det,physics.ins-det,To appear in the NSS-MIC 2009 conference proce...,,,on behalf of the CDF Collaboration,2010
2,1009.2633v1,Growth of epitaxially oriented Ag nanoislands ...,"Clean Si(111)-(7{x7) surfaces, followed by air...",2010-09-14 11:34:02+00:00,2010-09-14T11:34:02Z,"Anupam Roy, K. Bhattacharjee, J. Ghatak, B. N....",http://arxiv.org/pdf/1009.2633v1,cond-mat.other,cond-mat.other,10 figures,,10.1016/j.apsusc.2011.09.060,,2010
3,1007.3718v2,MASSCLEANage -- Stellar Cluster Ages from Inte...,We present the recently updated and expanded M...,2010-07-21 18:53:24+00:00,2010-09-14T15:16:44Z,"Bogdan Popescu, M. M. Hanson",http://arxiv.org/pdf/1007.3718v2,astro-ph.GA,astro-ph.GA,"17 pages, 11 figures. Accepted for publication...",,10.1088/0004-637X/724/1/296,,2010
4,1010.2910v1,Characterizations of Abel-Grassmann's groupoid...,"In this paper, we have introduced the concept ...",2010-10-14 13:30:48+00:00,2010-10-14T13:30:48Z,"Madad Khan, Faisal",http://arxiv.org/pdf/1010.2910v1,math.GR,math.GR,,,,,2010


In [53]:
df.to_csv(ARXIV_RAW_DIR / 'arxiv_articles_2010-2025.csv', index=False)

<!-- structured-notebook -->
## Preprint Retrieval Note
The final sections adapt the same general idea to bioRxiv and medRxiv, using source-specific search syntax and pagination logic.


## bioRxiv

***
The CODE below searches for keywords within only TITLE and ABSTRACT of the article's JSON.
***

In [10]:
import requests, time, pandas as pd

#THESE are keywords by which bioRxiv,medRxiv searches within its articles
PHRASES = [
    "healthy aging","healthy ageing",
    "anti-aging","anti-ageing","anti aging","anti ageing",
    "biohacking","living longer",
    "well aging","well ageing","aging well","ageing well",
    "slow aging","slow ageing",
    "health rejuvenation","nootropics",
]

#This function build a query for search
def build_query(server="bioRxiv", y0=2010, y1=2025, use_synonyms=False):
    or_block      = " OR ".join(f'"{p}"' for p in PHRASES) #joins articles using OR statement, e.g. ("healthy aging" OR "healthy ageing" OR ...)
    q = f'(SRC:PPR) AND ("{server}") AND TITLE_ABS:({or_block}) AND (PUB_YEAR:[{y0} TO {y1}])' #concatenates the query with 
    params = {
        "query": q,
        "format": "json",
        "pageSize": 1000,      # max page size
        "resultType": "core",  # includes abstractText, pmid/pmcid when present
        "cursorMark": "*",     # start of cursor pagination
        "synonym": str(use_synonyms).lower()
    }
    return params

def epmc_search_all(params, pause=0.2, max_pages=None):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    rows, total, pages = [], None, 0
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        js = r.json()
        if total is None:
            total = js.get("hitCount")
            print(f"Reported hitCount: {total}") #how many articles found in total
        batch = js.get("resultList", {}).get("result", [])
        rows.extend(batch) # append results to "rows"
        next_cursor = js.get("nextCursorMark")
        if not batch or next_cursor == params["cursorMark"]:
            break
        params["cursorMark"] = next_cursor # paginate through results
        pages += 1
        if max_pages and pages >= max_pages: # if the number of pages is greater than the limit we set, break
            break
        time.sleep(pause) # wait between every fetch
    return rows

if __name__ == "__main__":
    params = build_query(server = "bioRxiv", y0=2010, y1=2025, use_synonyms=False) #HERE you can change "bioRxiv" to "medRxiv" to retreive from 2 different servers
    data   = epmc_search_all(params)
    df     = pd.json_normalize(data)

    keep = ["id","pmid","pmcid","doi","title","pubYear","abstractText","source","authorString","server"]
    keep = [c for c in keep if c in df.columns]
    df_subset = df.reindex(columns=keep)

    df_subset.to_csv(PREPRINT_RAW_DIR / "biorxiv_longevity_titleabs_2010_2025.csv", index=False)
    print(f"Saved {len(df_subset)} rows")


Reported hitCount: 641
Saved 1282 rows


***
THE CODE below searches for keywords in EVERY field of the retreived JSON.
***

In [4]:
import requests
import time
import pandas as pd

PHRASES = [
    "healthy aging","healthy ageing",
    "anti-aging","anti-ageing","anti aging","anti ageing",
    "biohacking","living longer",
    "well aging","well ageing","aging well","ageing well",
    "slow aging","slow ageing",
    "health rejuvenation","nootropics",
]

def build_query(server="bioRxiv", y0=2010, y1=2025):
    or_block = " OR ".join(f'"{p}"' for p in PHRASES)
    return f'(SRC:PPR) AND ("{server}") AND ({or_block}) AND (PUB_YEAR:[{y0} TO {y1}])'

def epmc_search_all(q, page_size=1000, pause=0.2):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    cursor = "*"
    rows, total = [], None
    while True:
        params = {
            "query": q,
            "format": "json",
            "pageSize": page_size,
            "cursorMark": cursor,
            "resultType": "core",    # Retrieve metadata for every article
            "synonym": "false"
        }
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        js = r.json()
        if total is None:
            total = js.get("hitCount")
            print(f"Reported hitCount: {total}")
        batch = js.get("resultList", {}).get("result", [])
        rows.extend(batch)
        next_cursor = js.get("nextCursorMark")
        if not batch or next_cursor == cursor:
            break
        cursor = next_cursor
        time.sleep(pause)
    return rows

if __name__ == "__main__":
    q = build_query(server="medRxiv", y0=2010, y1=2025)
    data = epmc_search_all(q)
    df = pd.json_normalize(data)
    
    keep = ["id", "pmid", "pmcid", "doi", "title", "pubYear", "abstractText", "source", "authorString"]
    df_subset = df.reindex(columns=[c for c in keep if c in df.columns])
    
    df_subset.to_csv(PREPRINT_RAW_DIR / "biorxiv_longevity_with_pmid.csv", index=False)
    print(f"Saved {len(df_subset)} rows (with PMID if present)")


Reported hitCount: 241
Saved 482 rows (with PMID if present)
